In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from sklearn.feature_selection import mutual_info_classif
from xgboost import XGBClassifier, XGBRegressor

from sklearn.ensemble import RandomForestRegressor

In [2]:
heart_df = pd.read_csv("../data/brfss_2023_heart_risk_clean.csv") 
heart_df

,ever_heart_attack,high_cholesterol,ever_smoked_100_cigs,exercise_past_30_days,age_group,sex,poor_physical_health_days,poor_mental_health_days,activity_limited_health_days,current_asthma,...,copd_history,kidney_disease_history,stroke_history,difficulty_walking,current_smoker,bmi,general_health_label,high_blood_pressure_label,diabetes_label,alcohol_days_month
0,0,0.0,0.0,0.0,45-49,Female,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,30.47,Very good,Yes,Yes,0.0
1,0,1.0,0.0,1.0,45-49,Female,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,28.56,Very good,Yes,No,0.0
2,0,1.0,1.0,1.0,45-49,Female,6.0,2.0,6.0,1.0,...,0.0,0.0,0.0,1.0,0.0,22.31,Fair,Yes,No,0.0
3,0,0.0,0.0,1.0,45-49,Female,2.0,0.0,2.0,0.0,...,0.0,0.0,0.0,1.0,0.0,27.44,Very good,No,No,0.0
4,0,0.0,0.0,1.0,45-49,Female,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,25.85,Fair,Yes,Yes,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
430750,0,1.0,0.0,1.0,45-49,Male,12.0,30.0,12.0,0.0,...,0.0,0.0,NaN,0.0,0.0,29.21,Good,Yes,No,22.0
430751,0,0.0,0.0,0.0,25-29,Female,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,24.96,Very good,No,No,0.0
430752,0,1.0,0.0,1.0,35-39,Female,10.0,0.0,10.0,0.0,...,0.0,0.0,0.0,0.0,0.0,34.38,Very good,No,No,NaN
430753,0,1.0,0.0,1.0,45-49,Female,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,23.86,Good,Yes,Yes,0.0


XGBClassifier                                     

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, recall_score, confusion_matrix, balanced_accuracy_score, accuracy_score, precision_score, f1_score
from sklearn.feature_selection import SelectFromModel

models_score = []

def evaluate_model(model_name,  probs, y_test, threshold=0.5):
    preds = (probs >= threshold).astype(int)      # convert to 0/1 based on threshold
    acc = accuracy_score(y_test, preds)
    acc_bal = balanced_accuracy_score(y_test, preds, adjusted=True)     
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    f2 = ((1 + 2**2) * prec * rec) / (2**2 * prec + rec)
    
    print("\n============================")
    print(model_name)
    print("============================")

    
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    print("Confusion Matrix:")
    print(f"TP={tp}, FN={fn}, FP={fp}, TN={tn}")
    print("Recall =", recall_score(y_test, preds))

    models_score.append({
            "Model": model_name,
            "Accuracy": acc,
            "BalancedAccuracy":acc_bal,
            "Precision": prec,
            "Recall": rec,
            "F1 Score": f1,
            "F2": f2
        })
    results_df = pd.DataFrame(models_score).sort_values(by="Recall", ascending=False).reset_index(drop=True)
    return results_df

In [4]:
X = heart_df.drop('ever_heart_attack', axis=1)
y = heart_df['ever_heart_attack']

X_train, X_test, y_train, y_test = train_test_split(         
    X, y, test_size=0.2, random_state=42, stratify=y
)  

In [10]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
#from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE

# --------------------------------------------------
# 1. LOAD DATA
# --------------------------------------------------
heart_df = pd.read_csv("../data/brfss_2023_heart_risk_clean.csv") 

# make minority classes binary, too
binary_condition_map = {
    "No": 0,
    "No, but Borderline": 0,
    "Yes, during pregnancy": 0,
    "Yes": 1
}
heart_df["diabetes_label"] = (
    heart_df["diabetes_label"]
    .map(binary_condition_map)
    .astype("Int64")
)

heart_df["high_blood_pressure_label"] = (
    heart_df["high_blood_pressure_label"]
    .map(binary_condition_map)
    .astype("Int64")
)


# Target
X = heart_df.drop('ever_heart_attack', axis=1)
y = heart_df['ever_heart_attack']
# --------------------------------------------------
# 7. TRAIN TEST SPLIT
# --------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --------------------------------------------------
# 2. IDENTIFY COLUMN TYPES
# --------------------------------------------------
categorical_cols = X.select_dtypes(include=['object']).columns
binary_cols = X.select_dtypes(include=['int64']).columns
numeric_cols = X.select_dtypes(include=['float64']).columns

# --------------------------------------------------
# 3. PREPROCESSING STEPS
# --------------------------------------------------
preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols),
        ("bin", SimpleImputer(strategy="constant"), binary_cols),
        ("num", SimpleImputer(strategy="mean"), numeric_cols)
    ]
)

# --------------------------------------------------
# 4. FEATURE SELECTION
# --------------------------------------------------
selector = SelectKBest(score_func=mutual_info_classif, k=20)

# --------------------------------------------------
# 5. XGBOOST CLASSIFIER (Good starting params)
# --------------------------------------------------
# Calculate imbalance ratio
# pos = sum(y_train == 1)
# neg = sum(y_train == 0)
# scale_pos_weight = neg / pos
# best params from other notebook:
model = XGBClassifier(
    n_estimators=656,
    learning_rate=0.016,
    max_depth=4,
    subsample=0.92,
    reg_alpha=4.57e-06,
    clf__reg_lambda=8.9e-08,
    colsample_bytree=0.446,
    gamma=0.869,
    min_child_weight=2,
    eval_metric="logloss",
    #scale_pos_weight=17.36#scale_pos_weight
)
# model = XGBClassifier(
#     n_estimators=300,
#     learning_rate=0.05,
#     max_depth=5,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     eval_metric="logloss",
#     #scale_pos_weight=scale_pos_weight
# )


# --------------------------------------------------
# 6. CREATE FULL PIPELINE
# --------------------------------------------------
pipeline = Pipeline([
    ("preprocess", preprocess),
    ("smote", SMOTE(random_state=42)),  # apply SMOTE after encoding
    ("select_features", selector),
    ("model", model)
])

# --------------------------------------------------
# 8. FIT PIPELINE
# --------------------------------------------------
pipeline.fit(X_train, y_train)

# --------------------------------------------------
# 9. EVALUATE
# --------------------------------------------------
#y_pred = pipeline.predict(X_test)

y_proba = pipeline.predict_proba(X_test)[:,1]
threshold = 0.5  # lower than 0.5 to catch more positives
y_pred_new = (y_proba >= threshold).astype(int)


print("\nCONFUSION MATRIX:\n", confusion_matrix(y_test, y_pred_new))
print("\nCLASSIFICATION REPORT:\n", classification_report(y_test, y_pred_new))

d:\Nadine\Dokumente\WBS_data_science\10_finalProject\LoveYourHeart\.venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [17:01:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "clf__reg_lambda" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



CONFUSION MATRIX:
 [[79448  2013]
 [ 4030   660]]

CLASSIFICATION REPORT:
               precision    recall  f1-score   support

           0       0.95      0.98      0.96     81461
           1       0.25      0.14      0.18      4690

    accuracy                           0.93     86151
   macro avg       0.60      0.56      0.57     86151
weighted avg       0.91      0.93      0.92     86151



In [16]:
y_proba = pipeline.predict_proba(X_test)[:,1]
evaluate_model("XGB", y_proba , y_test)


XGB
Confusion Matrix:
TP=660, FN=4030, FP=2013, TN=79448
Recall = 0.14072494669509594


,Model,Accuracy,BalancedAccuracy,Precision,Recall,F1 Score,F2
0,XGB,0.929856,0.116014,0.246914,0.140725,0.179275,0.153968


In [13]:
y_proba = pipeline.predict_proba(X_test)[:,1]
threshold = 0.2  # lower than 0.5 to catch more positives
y_pred_new = (y_proba >= threshold).astype(int)


print("\nCONFUSION MATRIX:\n", confusion_matrix(y_test, y_pred_new))
print("\nCLASSIFICATION REPORT:\n", classification_report(y_test, y_pred_new))


CONFUSION MATRIX:
 [[63995 17466]
 [ 1603  3087]]

CLASSIFICATION REPORT:
               precision    recall  f1-score   support

           0       0.98      0.79      0.87     81461
           1       0.15      0.66      0.24      4690

    accuracy                           0.78     86151
   macro avg       0.56      0.72      0.56     86151
weighted avg       0.93      0.78      0.84     86151



## Randomized search

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, recall_score, f1_score, roc_auc_score
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from scipy.stats import uniform, randint

# --------------------------------------------------
# 1. LOAD DATA
# --------------------------------------------------
heart_df = pd.read_csv("../data/brfss_2023_heart_risk_clean.csv") 

# Make minority classes binary
binary_condition_map = {
    "No": 0,
    "No, but Borderline": 0,
    "Yes, during pregnancy": 0,
    "Yes": 1
}
heart_df["diabetes_label"] = (
    heart_df["diabetes_label"]
    .map(binary_condition_map)
    .astype("Int64")
)
heart_df["high_blood_pressure_label"] = (
    heart_df["high_blood_pressure_label"]
    .map(binary_condition_map)
    .astype("Int64")
)

# Target
X = heart_df.drop('ever_heart_attack', axis=1)
y = heart_df['ever_heart_attack']

# --------------------------------------------------
# 2. TRAIN TEST SPLIT
# --------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --------------------------------------------------
# 3. IDENTIFY COLUMN TYPES
# --------------------------------------------------
categorical_cols = X.select_dtypes(include=['object']).columns
binary_cols = X.select_dtypes(include=['int64']).columns
numeric_cols = X.select_dtypes(include=['float64']).columns

# --------------------------------------------------
# 4. PREPROCESSING
# --------------------------------------------------
preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols),
        ("bin", SimpleImputer(strategy="constant", fill_value=0), binary_cols),
        ("num", SimpleImputer(strategy="mean"), numeric_cols)
    ]
)

# --------------------------------------------------
# 5. CREATE PIPELINE
# --------------------------------------------------
pipeline = Pipeline([
    ("preprocess", preprocess),
    ("smote", SMOTE(random_state=42)),
    ("select_features", SelectKBest(score_func=mutual_info_classif)),
    ("model", XGBClassifier(eval_metric="logloss", random_state=42))
])

# --------------------------------------------------
# 6. PARAMETER DISTRIBUTIONS (FOCUSED ON KEY PARAMS)
# --------------------------------------------------
param_distributions = {
    # SMOTE sampling strategy - how much to oversample minority class
    'smote__sampling_strategy': [0.3, 0.5, 0.7, 1.0],  # 1.0 = full balance
    
    # Feature selection - how many features to keep
    'select_features__k': randint(15, 40),
    
    # XGBoost - Most important for imbalanced data
    'model__n_estimators': randint(200, 800),
    'model__max_depth': randint(3, 8),
    'model__learning_rate': uniform(0.01, 0.15),
    
    # Regularization - prevents overfitting on minority class
    'model__min_child_weight': randint(1, 10),
    'model__gamma': uniform(0, 2),
    'model__reg_alpha': uniform(0, 5),
    'model__reg_lambda': uniform(0, 5),
    
    # Sampling parameters - helps with overfitting
    'model__subsample': uniform(0.6, 0.4),  # 0.6 to 1.0
    'model__colsample_bytree': uniform(0.4, 0.6),  # 0.4 to 1.0
    
    # Class imbalance handling
    'model__scale_pos_weight': [1, 5, 10, 15, 20]  # Adjust for class ratio
}

# --------------------------------------------------
# 7. CUSTOM SCORING FOR IMBALANCED DATA
# --------------------------------------------------
# For imbalanced data, we care about catching positives (recall)
# and overall discrimination (ROC-AUC)
scoring = {
    'roc_auc': 'roc_auc',
    'recall': make_scorer(recall_score),
    'f1': make_scorer(f1_score)
}

# --------------------------------------------------
# 8. RANDOMIZED SEARCH
# --------------------------------------------------
random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    n_iter=50,  # Try 50 random combinations
    scoring=scoring,
    refit='roc_auc',  # Optimize for ROC-AUC (good for imbalanced)
    cv=3,  # 3-fold CV (faster than 5-fold)
    verbose=2,
    random_state=42,
    n_jobs=-1  # Use all CPU cores
)

print("Starting Randomized Search...")
print(f"Training samples: {len(X_train)}")
print(f"Positive class ratio: {sum(y_train == 1) / len(y_train):.3f}")
print("-" * 60)

random_search.fit(X_train, y_train)

# --------------------------------------------------
# 9. RESULTS
# --------------------------------------------------
print("\n" + "="*60)
print("BEST PARAMETERS:")
print("="*60)
for param, value in random_search.best_params_.items():
    print(f"{param}: {value}")

print("\n" + "="*60)
print("BEST CROSS-VALIDATION SCORES:")
print("="*60)
print(f"ROC-AUC: {random_search.cv_results_['mean_test_roc_auc'][random_search.best_index_]:.4f}")
print(f"Recall:  {random_search.cv_results_['mean_test_recall'][random_search.best_index_]:.4f}")
print(f"F1:      {random_search.cv_results_['mean_test_f1'][random_search.best_index_]:.4f}")

# --------------------------------------------------
# 10. TEST SET EVALUATION
# --------------------------------------------------
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = random_search.predict(X_test)
y_proba = random_search.predict_proba(X_test)[:, 1]

print("\n" + "="*60)
print("TEST SET PERFORMANCE:")
print("="*60)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print(f"\nROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

# --------------------------------------------------
# 11. SAVE RESULTS
# --------------------------------------------------
results_df = pd.DataFrame(random_search.cv_results_)
results_df = results_df.sort_values('rank_test_roc_auc')
results_df[['params', 'mean_test_roc_auc', 'mean_test_recall', 'mean_test_f1']].head(10).to_csv(
    'random_search_top10.csv', index=False
)
print("\nTop 10 results saved to 'random_search_top10.csv'")

# Save best model
import pickle
with open('best_heart_attack_model.pkl', 'wb') as f:
    pickle.dump(random_search.best_estimator_, f)
print("Best model saved to 'best_heart_attack_model.pkl'")

Starting Randomized Search...
Training samples: 344604
Positive class ratio: 0.054
------------------------------------------------------------
Fitting 3 folds for each of 50 candidates, totalling 150 fits


d:\Nadine\Dokumente\WBS_data_science\10_finalProject\LoveYourHeart\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:782: UserWarning: k=37 is greater than n_features=31. All the features will be returned.
  warnings.warn(



BEST PARAMETERS:
model__colsample_bytree: 0.7248687842965396
model__gamma: 1.3915687986901644
model__learning_rate: 0.04428250326959495
model__max_depth: 4
model__min_child_weight: 9
model__n_estimators: 771
model__reg_alpha: 1.304145874152045
model__reg_lambda: 4.981268498789621
model__scale_pos_weight: 15
model__subsample: 0.8233173814428391
select_features__k: 37
smote__sampling_strategy: 0.3

BEST CROSS-VALIDATION SCORES:
ROC-AUC: 0.8364
Recall:  0.7956
F1:      0.2426

TEST SET PERFORMANCE:

Confusion Matrix:
[[58526 22935]
 [  877  3813]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.72      0.83     81461
           1       0.14      0.81      0.24      4690

    accuracy                           0.72     86151
   macro avg       0.56      0.77      0.54     86151
weighted avg       0.94      0.72      0.80     86151


ROC-AUC Score: 0.8419

Top 10 results saved to 'random_search_top10.csv'
Best model saved to 'be

MLPClassifier (Neural Networks)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer

# Load data
heart_df = pd.read_csv("../data/brfss_2023_heart_risk_clean.csv") 

# make minority classes binary, too
binary_condition_map = {
    "No": 0,
    "No, but Borderline": 0,
    "Yes, during pregnancy": 0,
    "Yes": 1
}
heart_df["diabetes_label"] = (
    heart_df["diabetes_label"]
    .map(binary_condition_map)
    .astype("Int64")
)

heart_df["high_blood_pressure_label"] = (
    heart_df["high_blood_pressure_label"]
    .map(binary_condition_map)
    .astype("Int64")
)

# Target
X = heart_df.drop('ever_heart_attack', axis=1)
y = heart_df['ever_heart_attack']

# --------------------------------------------------
# 2. IDENTIFY COLUMN TYPES
# --------------------------------------------------
categorical_cols = X.select_dtypes(include=['object']).columns
binary_cols = X.select_dtypes(include=['int64']).columns
numeric_cols = X.select_dtypes(include=['float64']).columns

# --------------------------------------------------
# 3. PREPROCESSING STEPS
# --------------------------------------------------
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("bin", SimpleImputer(strategy="constant"), binary_cols),
        ("num", SimpleImputer(strategy="mean"), numeric_cols)
    ]
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

#smote = SMOTE(random_state=42)
#X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Neural network
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128,64),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=1000,
    early_stopping=True,
    random_state=42
)

# Full pipeline with SMOTE
pipeline_nn = ImbPipeline([
    ("preprocess", preprocess),  # encodes categorical + scales numeric
    ("smote", SMOTE(random_state=42)),  # apply SMOTE after encoding
    ("mlp", mlp_model)
])

# Fit
pipeline_nn.fit(X_train, y_train)

# Predict
#y_pred = pipeline_nn.predict(X_test)
y_proba = pipeline_nn.predict_proba(X_test)[:,1]  # risk score / probability
y_pred = (y_proba >= 0.5).astype(int)  # example threshold to reduce false negatives

# Evaluate
from sklearn.metrics import classification_report, confusion_matrix
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Confusion Matrix:
 [[64937 16524]
 [ 1722  2968]]

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.80      0.88     81461
           1       0.15      0.63      0.25      4690

    accuracy                           0.79     86151
   macro avg       0.56      0.71      0.56     86151
weighted avg       0.93      0.79      0.84     86151

Confusion Matrix:
 [[64937 16524]
 [ 1722  2968]]

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.80      0.88     81461
           1       0.15      0.63      0.25      4690

    accuracy                           0.79     86151
   macro avg       0.56      0.71      0.56     86151
weighted avg       0.93      0.79      0.84     86151



Voting Classifier (RF + GB + XGB)

In [4]:
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Define individual models
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight='balanced')
gb = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=5)
xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5,
                    subsample=0.8, colsample_bytree=0.8,
                    eval_metric='logloss', use_label_encoder=False,
                    scale_pos_weight=(sum(y_train==0)/sum(y_train==1)))

# Voting Classifier (soft voting to average predicted probabilities)
voting_clf = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('xgb', xgb)],
    voting='soft',  # uses predicted probabilities
    n_jobs=-1
)

# Full pipeline
pipeline_voting = Pipeline([
    ("preprocess", preprocess),
    ("voting", voting_clf)
])

# Fit
pipeline_voting.fit(X_train, y_train)

# Predict probabilities and classes
y_proba = pipeline_voting.predict_proba(X_test)[:,1]  # risk score / probability
y_pred = (y_proba >= 0.2).astype(int)  # example threshold to reduce false negatives

# Evaluate
from sklearn.metrics import classification_report, confusion_matrix
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Confusion Matrix:
 [[43283 38178]
 [  311  4379]]

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.53      0.69     81461
           1       0.10      0.93      0.19      4690

    accuracy                           0.55     86151
   macro avg       0.55      0.73      0.44     86151
weighted avg       0.94      0.55      0.66     86151



Voting Classifier (Included LR also, since got godd prediction in previous)

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingClassifier, GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# ---------------------------
# 1. Load data
# ---------------------------
df = pd.read_csv("./data/heart_2020_cleaned.csv")
df['HeartDisease'] = df['HeartDisease'].map({'No':0, 'Yes':1})

X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

# ---------------------------
# 2. Column types
# ---------------------------
categorical_cols = X.select_dtypes(include=['object']).columns
numeric_cols = X.select_dtypes(include=['int64','float64']).columns

# ---------------------------
# 3. Preprocessing
# ---------------------------
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numeric_cols)
])

# ---------------------------
# 4. Train-test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ---------------------------
# 5. Define models
# ---------------------------
# Random Forest with class weight
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)

# Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5
)

# XGBoost with scale_pos_weight
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    use_label_encoder=False,
    scale_pos_weight=(sum(y_train==0)/sum(y_train==1)),
    random_state=42
)

# Logistic Regression with class weight
lr = LogisticRegression(
    max_iter=5000,
    class_weight='balanced',
    solver='liblinear'
)

# ---------------------------
# 6. Voting Classifier
# ---------------------------
voting_clf = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('xgb', xgb), ('lr', lr)],
    voting='soft',  # use probabilities
    n_jobs=-1
)

# ---------------------------
# 7. Full pipeline
# ---------------------------
pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('voting', voting_clf)
])

# ---------------------------
# 8. Train
# ---------------------------
pipeline.fit(X_train, y_train)

# ---------------------------
# 9. Predict probabilities and classes
# ---------------------------
y_proba = pipeline.predict_proba(X_test)[:,1]  # risk score (0-1)

# Threshold tuning: example threshold = 0.19
threshold = 0.19
y_pred = (y_proba >= threshold).astype(int)

# ---------------------------
# 10. Evaluate
# ---------------------------
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ---------------------------
# 11. Optional: Risk categories
# ---------------------------
def get_risk_category(prob):
    if prob < 0.2:
        return 'Low'
    elif prob < 0.5:
        return 'Moderate'
    else:
        return 'High'

risk_category = [get_risk_category(p) for p in y_proba]

# Add to dataframe if needed
df_result = X_test.copy()
df_result['RiskScore'] = y_proba
df_result['RiskCategory'] = risk_category
df_result['TrueLabel'] = y_test.values

df_result.head()


Confusion Matrix:
 [[27581 30903]
 [  289  5186]]

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.47      0.64     58484
           1       0.14      0.95      0.25      5475

    accuracy                           0.51     63959
   macro avg       0.57      0.71      0.44     63959
weighted avg       0.92      0.51      0.61     63959



,BMI,Smoking,AlcoholDrinking,Stroke,PhysicalHealth,MentalHealth,DiffWalking,Sex,AgeCategory,Race,Diabetic,PhysicalActivity,GenHealth,SleepTime,Asthma,KidneyDisease,SkinCancer,RiskScore,RiskCategory,TrueLabel
229267,27.12,No,No,No,30,7,No,Male,45-49,White,Yes,Yes,Fair,8,No,No,No,0.497268,Moderate,0
133926,49.39,Yes,No,No,25,2,Yes,Female,45-49,White,No,Yes,Fair,6,Yes,No,No,0.424516,Moderate,0
307338,32.92,No,No,No,3,29,No,Female,35-39,White,No,Yes,Good,8,Yes,No,No,0.127480,Low,0
284517,26.36,Yes,No,No,0,0,No,Male,70-74,White,No,Yes,Fair,7,No,No,No,0.682428,High,0
58968,22.71,No,No,No,0,10,No,Female,65-69,White,No,Yes,Excellent,7,No,No,No,0.101438,Low,0


In [ ]:
heart_df.loc[heart_df['HeartDisease'] == 1, :].sample(25)

,HeartDisease,BMI,Smoking,AlcoholDrinking,Stroke,PhysicalHealth,MentalHealth,DiffWalking,Sex,AgeCategory,Race,Diabetic,PhysicalActivity,GenHealth,SleepTime,Asthma,KidneyDisease,SkinCancer
5,1,28.87,Yes,No,No,6,0,Yes,Female,75-79,Black,No,No,Fair,12,No,No,No
10,1,34.30,Yes,No,No,30,0,Yes,Male,60-64,White,Yes,No,Poor,15,Yes,No,No
35,1,32.98,Yes,No,Yes,10,0,Yes,Male,75-79,White,Yes,Yes,Poor,4,No,No,Yes
42,1,25.06,No,No,No,0,0,Yes,Female,80 or older,White,Yes,No,Good,7,No,No,Yes
43,1,30.23,Yes,No,No,6,2,Yes,Female,75-79,White,Yes,Yes,Fair,8,No,Yes,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319765,1,38.45,No,No,Yes,30,15,Yes,Female,55-59,Hispanic,Yes,Yes,Poor,6,Yes,No,No
319767,1,36.21,Yes,No,No,0,0,Yes,Female,75-79,Hispanic,Yes,Yes,Good,8,No,No,No
319781,1,37.12,Yes,No,No,0,0,No,Male,35-39,Hispanic,No,Yes,Very good,7,No,No,No
319786,1,33.20,Yes,No,No,0,0,No,Female,60-64,Hispanic,Yes,Yes,Very good,8,Yes,No,No


with SMOTE to handle imbalance

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# ---------------------------
# 1. Load data
# ---------------------------
df = pd.read_csv("./data/heart_2020_cleaned.csv")
df['HeartDisease'] = df['HeartDisease'].map({'No':0, 'Yes':1})

X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

# ---------------------------
# 2. Column types
# ---------------------------
categorical_cols = X.select_dtypes(include=['object']).columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns

# ---------------------------
# 3. Preprocessing
# ---------------------------
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numeric_cols)
])

# ---------------------------
# 4. Train-test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ---------------------------
# 5. Handle imbalance with SMOTE
# ---------------------------
smote = SMOTE(random_state=42)

# ---------------------------
# 6. Define base models with best parameters observed
# ---------------------------
rf = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    random_state=42,
    class_weight='balanced'
)

gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5
)

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    use_label_encoder=False,
    scale_pos_weight=(sum(y_train==0)/sum(y_train==1)),
    random_state=42
)

lr = LogisticRegression(
    max_iter=5000,
    class_weight='balanced',
    solver='liblinear'
)

# ---------------------------
# 7. Voting Classifier (soft voting)
# ---------------------------
voting_clf = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('xgb', xgb), ('lr', lr)],
    voting='soft',
    n_jobs=-1,
    weights=[1,2,2,1]  # give more weight to boosting models
)

# ---------------------------
# 8. Full pipeline
# ---------------------------
pipeline = ImbPipeline([
    ('preprocess', preprocessor),
    ('smote', smote),
    ('voting', voting_clf)
])

# ---------------------------
# 9. Train
# ---------------------------
pipeline.fit(X_train, y_train)

# ---------------------------
# 10. Predict probabilities and classes
# ---------------------------
y_proba = pipeline.predict_proba(X_test)[:,1]  # risk score (0-1)

# Tunable threshold to balance recall vs precision
threshold = 0.25
y_pred = (y_proba >= threshold).astype(int)

# ---------------------------
# 11. Evaluate
# ---------------------------
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ---------------------------
# 12. Risk categories for app
# ---------------------------
def get_risk_category(prob):
    if prob < 0.2:
        return 'Low'
    elif prob < 0.5:
        return 'Moderate'
    else:
        return 'High'

risk_category = [get_risk_category(p) for p in y_proba]

# ---------------------------
# 13. Create final output DataFrame
# ---------------------------
df_result = X_test.copy()
df_result['RiskScore'] = y_proba
df_result['RiskCategory'] = risk_category
df_result['PredictedLabel'] = y_pred
df_result['TrueLabel'] = y_test.values

df_result.head()


Confusion Matrix:
 [[27853 30631]
 [  344  5131]]

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.48      0.64     58484
           1       0.14      0.94      0.25      5475

    accuracy                           0.52     63959
   macro avg       0.57      0.71      0.45     63959
weighted avg       0.92      0.52      0.61     63959



,BMI,Smoking,AlcoholDrinking,Stroke,PhysicalHealth,MentalHealth,DiffWalking,Sex,AgeCategory,Race,...,PhysicalActivity,GenHealth,SleepTime,Asthma,KidneyDisease,SkinCancer,RiskScore,RiskCategory,PredictedLabel,TrueLabel
229267,27.12,No,No,No,30,7,No,Male,45-49,White,...,Yes,Fair,8,No,No,No,0.568979,High,1,0
133926,49.39,Yes,No,No,25,2,Yes,Female,45-49,White,...,Yes,Fair,6,Yes,No,No,0.395380,Moderate,1,0
307338,32.92,No,No,No,3,29,No,Female,35-39,White,...,Yes,Good,8,Yes,No,No,0.102897,Low,0,0
284517,26.36,Yes,No,No,0,0,No,Male,70-74,White,...,Yes,Fair,7,No,No,No,0.824912,High,1,0
58968,22.71,No,No,No,0,10,No,Female,65-69,White,...,Yes,Excellent,7,No,No,No,0.164296,Low,0,0


In [21]:
heart_df.columns

Index(['HeartDisease', 'BMI', 'Smoking', 'AlcoholDrinking', 'Stroke',
       'PhysicalHealth', 'MentalHealth', 'DiffWalking', 'Sex', 'AgeCategory',
       'Race', 'Diabetic', 'PhysicalActivity', 'GenHealth', 'SleepTime',
       'Asthma', 'KidneyDisease', 'SkinCancer'],
      dtype='object')

Feature Engineering 

In [ ]:
heartFea_df = heart_df.copy()

# Mapping columns with 'Yes' or 'No' to 0 and  1
binary_cols = ['Smoking','AlcoholDrinking','DiffWalking','PhysicalActivity','Asthma','KidneyDisease','SkinCancer','Stroke']
for col in binary_cols:
    heartFea_df[col] = heartFea_df[col].map({'Yes':1, 'No':0})
heartFea_df['LifestyleScore'] = heartFea_df[binary_cols].sum(axis=1)

# Total Health Score
heartFea_df['HealthScore'] = heartFea_df['PhysicalHealth'] + heartFea_df['MentalHealth']  # higher = worse
heartFea_df["SleepAdequate"] = heartFea_df["SleepTime"].apply(
    lambda x: 0 if x >= 8 else 1
)

# Diabetes numeric: No=0, Borderline=1, Yes=2, Yes during pregnancy=2
heartFea_df['DiabetesNum'] = heartFea_df['Diabetic'].map({'No':0,'No, borderline diabetes':1,'Yes (during pregnancy)':1,'Yes':2})
# General health numeric: Poor=1, Fair=2, Good=3, Very good=4, Excellent=5
heartFea_df['GenHealthNum'] = heartFea_df['GenHealth'].map({'Poor':4,'Fair':3,'Good':2,'Very good':1,'Excellent':0})

# --- AGE CATEGORY SCORING ---
age_map = {
    "18-24": 1,
    "25-29": 2,
    "30-34": 3,
    "35-39": 4,
    "40-44": 5,
    "45-49": 6,
    "50-54": 7,
    "55-59": 8,
    "60-64": 9,
    "65-69": 10,
    "70-74": 11,
    "75-79": 12,
    "80 or older": 13
}

# Create AgeScore
heartFea_df["AgeScore"] = heartFea_df["AgeCategory"].map(age_map)

# --- BMI * AGE INTERACTION ---
heartFea_df["BMI_AgeInteraction"] = heartFea_df["BMI"] * heartFea_df["AgeScore"]


,HeartDisease,BMI,Smoking,AlcoholDrinking,Stroke,PhysicalHealth,MentalHealth,DiffWalking,Sex,AgeCategory,Race,Diabetic,PhysicalActivity,GenHealth,SleepTime,Asthma,KidneyDisease,SkinCancer,LifestyleScore
0,0,16.60,1,0,0,3,30,0,Female,55-59,White,Yes,1,Very good,5,1,0,1,4
1,0,20.34,0,0,1,0,0,0,Female,80 or older,White,No,1,Very good,7,0,0,0,2
2,0,26.58,1,0,0,20,30,0,Male,65-69,White,Yes,1,Fair,8,1,0,0,3
3,0,24.21,0,0,0,0,0,0,Female,75-79,White,No,0,Good,6,0,0,1,1
4,0,23.71,0,0,0,28,0,1,Female,40-44,White,No,1,Very good,8,0,0,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319790,1,27.41,1,0,0,7,0,1,Male,60-64,Hispanic,Yes,0,Fair,6,1,0,0,3
319791,0,29.84,1,0,0,0,0,0,Male,35-39,Hispanic,No,1,Very good,5,1,0,0,3
319792,0,24.24,0,0,0,0,0,0,Female,45-49,Hispanic,No,1,Good,6,0,0,0,1
319793,0,32.81,0,0,0,0,0,0,Female,25-29,Hispanic,No,0,Good,12,0,0,0,0


In [39]:
heartFea_df

,HeartDisease,BMI,Smoking,AlcoholDrinking,Stroke,PhysicalHealth,MentalHealth,DiffWalking,Sex,AgeCategory,...,Asthma,KidneyDisease,SkinCancer,LifestyleScore,HealthScore,SleepAdequate,DiabetesNum,GenHealthNum,AgeScore,BMI_AgeInteraction
0,0,16.60,1,0,0,3,30,0,Female,55-59,...,1,0,1,4,33,1,2,1,8,132.80
1,0,20.34,0,0,1,0,0,0,Female,80 or older,...,0,0,0,2,0,1,0,1,13,264.42
2,0,26.58,1,0,0,20,30,0,Male,65-69,...,1,0,0,3,50,0,2,3,10,265.80
3,0,24.21,0,0,0,0,0,0,Female,75-79,...,0,0,1,1,0,1,0,2,12,290.52
4,0,23.71,0,0,0,28,0,1,Female,40-44,...,0,0,0,2,28,0,0,1,5,118.55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319790,1,27.41,1,0,0,7,0,1,Male,60-64,...,1,0,0,3,7,1,2,3,9,246.69
319791,0,29.84,1,0,0,0,0,0,Male,35-39,...,1,0,0,3,0,1,0,1,4,119.36
319792,0,24.24,0,0,0,0,0,0,Female,45-49,...,0,0,0,1,0,1,0,2,6,145.44
319793,0,32.81,0,0,0,0,0,0,Female,25-29,...,0,0,0,0,0,0,0,2,2,65.62


Voting Classifier with feature engineered data

In [40]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# ---------------------------
# 1. Load data
# ---------------------------
#df = pd.read_csv("./data/heart_2020_cleaned.csv")
#df['HeartDisease'] = df['HeartDisease'].map({'No':0, 'Yes':1})

X = heartFea_df.drop("HeartDisease", axis=1)
y = heartFea_df["HeartDisease"]

# ---------------------------
# 2. Column types
# ---------------------------
categorical_cols = X.select_dtypes(include=['object']).columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns

# ---------------------------
# 3. Preprocessing
# ---------------------------
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numeric_cols)
])

# ---------------------------
# 4. Train-test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ---------------------------
# 5. Handle imbalance with SMOTE
# ---------------------------
smote = SMOTE(random_state=42)

# ---------------------------
# 6. Define base models with best parameters observed
# ---------------------------
rf = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    random_state=42,
    class_weight='balanced'
)

gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5
)

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    use_label_encoder=False,
    scale_pos_weight=(sum(y_train==0)/sum(y_train==1)),
    random_state=42
)

lr = LogisticRegression(
    max_iter=5000,
    class_weight='balanced',
    solver='liblinear'
)

# ---------------------------
# 7. Voting Classifier (soft voting)
# ---------------------------
voting_clf = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('xgb', xgb), ('lr', lr)],
    voting='soft',
    n_jobs=-1,
    weights=[1,2,2,1]  # give more weight to boosting models
)

# ---------------------------
# 8. Full pipeline
# ---------------------------
pipeline = ImbPipeline([
    ('preprocess', preprocessor),
    ('smote', smote),
    ('voting', voting_clf)
])

# ---------------------------
# 9. Train
# ---------------------------
pipeline.fit(X_train, y_train)

# ---------------------------
# 10. Predict probabilities and classes
# ---------------------------
y_proba = pipeline.predict_proba(X_test)[:,1]  # risk score (0-1)

# Tunable threshold to balance recall vs precision
threshold = 0.25
y_pred = (y_proba >= threshold).astype(int)

# ---------------------------
# 11. Evaluate
# ---------------------------
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ---------------------------
# 12. Risk categories for app
# ---------------------------
def get_risk_category(prob):
    if prob < 0.2:
        return 'Low'
    elif prob < 0.5:
        return 'Moderate'
    else:
        return 'High'

risk_category = [get_risk_category(p) for p in y_proba]

# ---------------------------
# 13. Create final output DataFrame
# ---------------------------
df_result = X_test.copy()
df_result['RiskScore'] = y_proba
df_result['RiskCategory'] = risk_category
df_result['PredictedLabel'] = y_pred
df_result['TrueLabel'] = y_test.values

df_result.head()


Confusion Matrix:
 [[30864 27620]
 [  437  5038]]

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.53      0.69     58484
           1       0.15      0.92      0.26      5475

    accuracy                           0.56     63959
   macro avg       0.57      0.72      0.48     63959
weighted avg       0.91      0.56      0.65     63959



,BMI,Smoking,AlcoholDrinking,Stroke,PhysicalHealth,MentalHealth,DiffWalking,Sex,AgeCategory,Race,...,HealthScore,SleepAdequate,DiabetesNum,GenHealthNum,AgeScore,BMI_AgeInteraction,RiskScore,RiskCategory,PredictedLabel,TrueLabel
229267,27.12,0,0,0,30,7,0,Male,45-49,White,...,37,0,2,3,6,162.72,0.455793,Moderate,1,0
133926,49.39,1,0,0,25,2,1,Female,45-49,White,...,27,1,0,3,6,296.34,0.359042,Moderate,1,0
307338,32.92,0,0,0,3,29,0,Female,35-39,White,...,32,0,0,2,4,131.68,0.099948,Low,0,0
284517,26.36,1,0,0,0,0,0,Male,70-74,White,...,0,1,0,3,11,289.96,0.849296,High,1,0
58968,22.71,0,0,0,0,10,0,Female,65-69,White,...,10,1,0,0,10,227.10,0.099254,Low,0,0


MLPClassifier with feature engineered data

In [41]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Load data
#df = pd.read_csv("./data/heart_2020_cleaned.csv")
#df['HeartDisease'] = df['HeartDisease'].map({'No':0, 'Yes':1})

X = heartFea_df.drop("HeartDisease", axis=1)
y = heartFea_df["HeartDisease"]

# Column types
categorical_cols = X.select_dtypes(include=['object']).columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols)
    ]
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

#smote = SMOTE(random_state=42)
#X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Neural network
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128,64),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=1000,
    early_stopping=True,
    random_state=42
)

# Full pipeline with SMOTE
pipeline_nn = ImbPipeline([
    ("preprocess", preprocessor),  # encodes categorical + scales numeric
    ("smote", SMOTE(random_state=42)),  # apply SMOTE after encoding
    ("mlp", mlp_model)
])

# Fit
pipeline_nn.fit(X_train, y_train)

# Predict
#y_pred = pipeline_nn.predict(X_test)
y_proba = pipeline_nn.predict_proba(X_test)[:,1]  # risk score / probability
y_pred = (y_proba >= 0.2).astype(int)  # example threshold to reduce false negatives

# Evaluate
from sklearn.metrics import classification_report, confusion_matrix
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Confusion Matrix:
 [[41457 17027]
 [ 1570  3905]]

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.71      0.82     58484
           1       0.19      0.71      0.30      5475

    accuracy                           0.71     63959
   macro avg       0.58      0.71      0.56     63959
weighted avg       0.90      0.71      0.77     63959



Stacked average predictions with both (voting + nn)

In [44]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE

# -------------------------------
# 1. LOAD FEATURE-ENGINEERED DATA
# -------------------------------
# df = pd.read_csv("./data/heart_feature_engineered.csv")
# df['HeartDisease'] = df['HeartDisease'].map({'No':0, 'Yes':1})

X = heartFea_df.drop("HeartDisease", axis=1)
y = heartFea_df["HeartDisease"]

# -------------------------------
# 2. IDENTIFY COLUMN TYPES
# -------------------------------
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# -------------------------------
# 3. PREPROCESSING
# -------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols)
    ]
)

# -------------------------------
# 4. TRAIN-TEST SPLIT
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# -------------------------------
# 5. APPLY PREPROCESSING BEFORE SMOTE
# -------------------------------
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Apply SMOTE on transformed data
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_transformed, y_train)

# -------------------------------
# 6. DEFINE MODELS
# -------------------------------
# Neural Network
nn_model = MLPClassifier(
    hidden_layer_sizes=(128,64),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=1000,
    early_stopping=True,
    random_state=42
)

# Voting Ensemble: Logistic + Gradient Boosting + Random Forest
voting_model = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(class_weight='balanced', max_iter=5000, solver='liblinear')),
        ('gb', GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=5)),
        ('rf', RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced'))
    ],
    voting='soft'
)

# -------------------------------
# 7. FIT MODELS
# -------------------------------
nn_model.fit(X_train_res, y_train_res)
voting_model.fit(X_train_res, y_train_res)

# -------------------------------
# 8. STACKED PREDICTIONS
# -------------------------------
# Average the predicted probabilities
nn_probs = nn_model.predict_proba(X_test_transformed)[:,1]
voting_probs = voting_model.predict_proba(X_test_transformed)[:,1]
stacked_probs = (nn_probs + voting_probs) / 2

# Apply custom threshold to balance recall & precision
threshold = 0.19
stacked_pred = (stacked_probs >= threshold).astype(int)

# -------------------------------
# 9. EVALUATION
# -------------------------------
print("Confusion Matrix:\n", confusion_matrix(y_test, stacked_pred))
print("\nClassification Report:\n", classification_report(y_test, stacked_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, stacked_probs))

# -------------------------------
# 10. RISK CATEGORY
# -------------------------------
def get_risk_category(prob):
    if prob < 0.2:
        return "Low Risk"
    elif prob < 0.5:
        return "Moderate Risk"
    else:
        return "High Risk"

risk_categories = [get_risk_category(p) for p in stacked_probs]

# -------------------------------
# 11. FINAL OUTPUT
# -------------------------------
output_df = X_test.copy()
output_df['Predicted_Probability'] = stacked_probs
output_df['Predicted_Label'] = stacked_pred
output_df['Risk_Category'] = risk_categories

print(output_df.head())


Confusion Matrix:
 [[35883 22601]
 [  743  4732]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.61      0.75     58484
           1       0.17      0.86      0.29      5475

    accuracy                           0.64     63959
   macro avg       0.58      0.74      0.52     63959
weighted avg       0.91      0.64      0.71     63959

ROC-AUC Score: 0.8162512261871413
          BMI  Smoking  AlcoholDrinking  Stroke  PhysicalHealth  MentalHealth  \
229267  27.12        0                0       0              30             7   
133926  49.39        1                0       0              25             2   
307338  32.92        0                0       0               3            29   
284517  26.36        1                0       0               0             0   
58968   22.71        0                0       0               0            10   

        DiffWalking     Sex AgeCategory   Race  ... LifestyleScore  \
229267 

In [45]:
output_df

,BMI,Smoking,AlcoholDrinking,Stroke,PhysicalHealth,MentalHealth,DiffWalking,Sex,AgeCategory,Race,...,LifestyleScore,HealthScore,SleepAdequate,DiabetesNum,GenHealthNum,AgeScore,BMI_AgeInteraction,Predicted_Probability,Predicted_Label,Risk_Category
229267,27.12,0,0,0,30,7,0,Male,45-49,White,...,1,37,0,2,3,6,162.72,0.344496,1,Moderate Risk
133926,49.39,1,0,0,25,2,1,Female,45-49,White,...,4,27,1,0,3,6,296.34,0.208380,1,Moderate Risk
307338,32.92,0,0,0,3,29,0,Female,35-39,White,...,2,32,0,0,2,4,131.68,0.045480,0,Low Risk
284517,26.36,1,0,0,0,0,0,Male,70-74,White,...,2,0,1,0,3,11,289.96,0.873426,1,High Risk
58968,22.71,0,0,0,0,10,0,Female,65-69,White,...,1,10,1,0,0,10,227.10,0.053220,0,Low Risk
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233107,29.21,1,0,0,2,10,1,Male,60-64,White,...,3,12,1,2,1,9,262.89,0.516639,1,High Risk
169205,30.41,0,0,0,0,0,0,Male,60-64,White,...,1,0,1,0,0,9,273.69,0.307331,1,Moderate Risk
47453,38.52,0,0,0,2,0,0,Male,60-64,Black,...,2,2,0,0,3,9,346.68,0.581375,1,High Risk
294418,33.00,1,0,0,2,0,0,Male,70-74,Other,...,3,2,0,0,2,11,363.00,0.391569,1,Moderate Risk
